# Importance Sampling for observable estimation in NetKet

This notebook is a tutorial on the `expect_is` routine implemented in
`netket_pro/packages/spin_vmc/expectation_value/importance_sampling.py`.

## Motivation

In Variational Monte Carlo we estimate expectation values

$$
\langle O \rangle_\psi = \frac{\langle \psi | \hat O | \psi\rangle}{\langle \psi | \psi \rangle}
= \sum_\sigma \frac{|\psi(\sigma)|^2}{\lVert\psi\rVert^2}\, O_\text{loc}(\sigma),
\qquad
O_\text{loc}(\sigma) = \sum_\eta \langle\sigma|\hat O|\eta\rangle \frac{\psi(\eta)}{\psi(\sigma)},
$$

by drawing MCMC samples $\sigma_i \sim |\psi(\sigma)|^2$ and averaging $O_\text{loc}(\sigma_i)$.

The expensive step is the sampling itself: the MCMC chain must burn in and produce
enough decorrelated configurations. In a **Foundational NQS** setting, a single
model produces wavefunctions $\psi(\sigma; h_0)$ for every value of the Hamiltonian
parameter $h_0$. Evaluating observables across a fine sweep of $h_0$ means
re-running a full MCMC at every single point — even though nearby $h_0$ values
give very similar distributions $|\psi(\sigma; h_0)|^2$.

**Importance Sampling (IS)** avoids that waste. We sample MCMC at a few **anchor**
values of $h_0$ and then **reweight** those samples to estimate observables at
neighbouring points without re-sampling. For large systems where MCMC burn-in
dominates the cost, this gives a speedup roughly equal to the ratio of sweep points
to anchors.

 The only cost per point is a forward pass
through the ViT to evaluate $\log\psi_\text{tgt}$ on the stored samples, plus the
connected-element sums for $O_\text{loc}$.

## 1. Theoretical recap

### 1.1 Reweighting identity

We want $\langle O\rangle$ for a **target** state $\psi_\text{tgt}$, but we only
have samples from a **reference** state $\psi_\text{ref}$. Inserting
$|\psi_\text{ref}|^2 / |\psi_\text{ref}|^2$:

$$
\langle O\rangle_\text{tgt}
= \frac{\sum_\sigma |\psi_\text{ref}(\sigma)|^2\, w(\sigma)\, O_\text{loc}^\text{tgt}(\sigma)}
        {\sum_\sigma |\psi_\text{ref}(\sigma)|^2\, w(\sigma)},
\qquad
w(\sigma) = \frac{|\psi_\text{tgt}(\sigma)|^2}{|\psi_\text{ref}(\sigma)|^2}.
$$

Both normalization constants cancel in the ratio. Monte-Carlo-averaging over
$\sigma_i \sim |\psi_\text{ref}|^2$ yields the **self-normalized IS** estimator:

$$
\widehat{\langle O\rangle}_\text{tgt}
= \frac{\sum_i w_i\, O_\text{loc}^\text{tgt}(\sigma_i)}{\sum_i w_i},
\qquad
w_i = \frac{|\psi_\text{tgt}(\sigma_i)|^2}{|\psi_\text{ref}(\sigma_i)|^2}.
$$



## 2. The Effective Sample Size (ESS)

The standard diagnostic for IS quality is the **Effective Sample Size**:

$$
\text{ESS} = \frac{\left(\sum_i w_i\right)^2}{\sum_i w_i^2},
\qquad 1 \le \text{ESS} \le N.
$$

ESS is the number of *equally-weighted* samples that would give
the same statistical power as the $N$ unevenly-weighted samples we actually have.

| ESS / N | Meaning |
|---------|:--------|
| $\approx 1$ | All weights equal. $\psi_\text{tgt} = \psi_\text{ref}$. Perfect reuse. |
| $\gtrsim 0.1$ | Reasonable overlap. IS estimate is trustworthy. |
| $\lesssim 0.01$ | A handful of samples dominate. Estimator is degenerate. |


 `expect_is` reports
$\sigma_{\bar O} = \sqrt{\mathrm{Var}_w(O_\text{loc}) / \text{ESS}}$,
so when the ESS collapses the error bar explodes accordingly.

## 3. The `expect_is` routine

```python
from spin_vmc.expectation_value import expect_is, ISResult

result = expect_is(operator, mc_ref, target_variables, chunk_size=None)
# result.stats         -> netket.stats.Stats (mean, error_of_mean, variance)
# result.ess           -> Effective Sample Size (float)
# result.ess_fraction  -> ESS / n_samples
# result.n_samples     -> total number of reference samples
```

Usage notes:

- `mc_ref` must be an `nk.vqs.MCState` **already sampled** (`mc_ref.sample()` called
  beforehand). `expect_is` reads `mc_ref.samples` directly.
- `target_variables` is a Flax-style dict that the **same model** can consume. In the
  FNQS workflow: `mc_ref = vs.get_state(h_ref)` gives a reference MCState, and
  `vs.get_state(h_tgt).variables` gives the target variables. The `_apply_fun` is
  shared, only the `foundational.parameters` key differs.
- The operator can be any NetKet discrete operator (diagonal or non-diagonal).
- `chunk_size` controls peak memory for the amplitude evaluation.

## 4. Concrete example: FNQS + importance sampling sweep

We will :

1. Train a small **Foundational NQS** (ViTFNQS) on the 1D transverse-field Ising
   model across a range of $h_0$ values.
2. After training, pick a few **anchor** values of $h_0$ where we run full MCMC.
3. For all other $h_0$ in a fine sweep, estimate $\langle H\rangle$ and
   $\langle M_z^2\rangle$ via IS from the nearest anchor.
4. **Compare** with the brute-force baseline that re-samples at every $h_0$.


### 4.1 Imports



In [ ]:
import os
os.environ["NETKET_EXPERIMENTAL_SHARDING"] = "1"
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import time
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import netket as nk
import netket_foundation as nkf
from netket_foundation._src.model.vit import ViTFNQS
from netket_foundation.expectation_value import expect_is

### 4.2 Physical system

1D transverse-field Ising model on $L=10$ spins with periodic boundary conditions:

$$
\hat H(h_0) = -\sum_{\langle ij\rangle} \sigma^z_i \sigma^z_j
              - h_0 \sum_i \sigma^x_i.
$$


In [ ]:
L = 10
hi = nk.hilbert.Spin(0.5, L)
graph = nk.graph.Chain(L, pbc=True)

# Parameter space: 1 parameter h0 in [0.8, 1.2].
ps = nkf.ParameterSpace(N=1, min=0.8, max=1.2)

def create_operator(params):
    """Build the Ising Hamiltonian for a given 1D array of parameters."""
    assert params.shape == (1,)
    h = params[0]
    ha_X = sum(nkf.operator.sigmax(hi, i) for i in range(L))
    ha_ZZ = sum(
        nkf.operator.sigmaz(hi, i) @ nkf.operator.sigmaz(hi, (i + 1) % L)
        for i in range(L)
    )
    return -h * ha_X - ha_ZZ

# Parametrised operator for the FNQS driver.
ha_p = nkf.operator.ParametrizedOperator(hi, ps, create_operator)

# Magnetization operators (parameter-independent).
# PauliStringsJax does not support division, so we multiply by (1/L).
Mz = sum(nkf.operator.sigmaz(hi, i) for i in range(L)) * (1.0 / L)
Mz2 = Mz @ Mz

print(f"System: 1D Ising, L={L}, h0 in [{ps._min}, {ps._max}]")

### 4.3 Train the Foundational NQS

We use a small ViTFNQS (2 layers, $d_\text{model}=12$, 4 heads) trained with the
natural-gradient VMC driver (`VMC_NG`) across 8 replicas, each at a different
$h_0$ spread uniformly in $[0.8, 1.2]$.

After training, `vs.get_state(pars)` returns an `MCState` for any $h_0$ value in
the parameter space — this is what we will pass to `expect_is`.


In [ ]:
import optax

# --- model ---
ma = ViTFNQS(
    num_layers=2,
    d_model=12,
    heads=4,
    L_eff=L // 2,        # patch size: 2 spins per patch
    n_coups=ps.size,     # number of Hamiltonian parameters
    b=2,
    complex=False,
    disorder=False,
    transl_invariant=True,
    two_dimensional=False,
)

# --- sampler & foundational state ---
sa = nk.sampler.MetropolisLocal(hi, n_chains=512)
vs = nkf.FoundationalQuantumState(sa, ma, ps, n_replicas=8, seed=1)

# Distribute the 8 replicas evenly across the parameter range.
vs.parameter_array = jnp.linspace(0.8, 1.2, vs.n_replicas).reshape(-1, 1)

# --- training ---
N_ITER = 300
optimizer = optax.sgd(0.01)
gs = nkf.VMC_NG(ha_p, optimizer, variational_state=vs, diag_shift=1e-4)

t0 = time.perf_counter()
gs.run(N_ITER, show_progress=True)
t_train = time.perf_counter() - t0
print(f"Training done in {t_train:.0f} s")

### 4.4 Choose anchors and sample them

We pick **3 anchor points** ($h_0 = 0.85, 1.0, 1.15$) and run full MCMC at each
one. These will be the *reference states* for IS. All other $h_0$ values in the
fine sweep will borrow samples from their nearest anchor.

For each anchor we call `vs.get_state(h0)` to get an `MCState`, then
`mc.sample()` to draw a fresh batch of MCMC samples. This is the only MCMC we pay.



In [ ]:
# Anchor h0 values where we do full MCMC.
h_anchors = np.array([0.85, 1.0, 1.15])

# Fine sweep: 21 points across the full range.
h_sweep = np.linspace(0.80, 1.20, 21)

# Build and sample anchor MCStates.
anchor_states = {}
for h0 in h_anchors:
    pars = jnp.array([h0])
    mc = vs.get_state(pars)
    mc.sample()  # full burn-in + MCMC chain
    anchor_states[h0] = mc
    print(f"  anchor h0={h0:.2f}  samples: {mc.samples.shape}")

def nearest_anchor(h, anchors=h_anchors):
    """Return the anchor h0 closest to h."""
    return anchors[np.argmin(np.abs(anchors - h))]

### 4.5 IS sweep from anchors

For each $h_0$ in the fine sweep we:
- Find the nearest anchor.
- Build the target `MCState` via `vs.get_state(h0)` to get its `.variables`.
- Call `expect_is(H, mc_anchor, target_vars)` to estimate $\langle H(h_0)\rangle$
   and $\langle M_z^2\rangle$ using the anchor's samples reweighted by
   $w_i = |\psi(h_0)|^2 / |\psi(h_\text{anchor})|^2$.




In [ ]:
# --- warm-up: compile expect_is with one throwaway call per anchor ---
for h0, mc in anchor_states.items():
    _w = expect_is(create_operator(jnp.array([h0])), mc, mc.variables)
    jax.block_until_ready(_w.stats.mean)

is_E, is_E_err, is_Mz2, is_Mz2_err = [], [], [], []
is_ess_frac, is_anchor_used = [], []

for h0 in h_sweep:
    # 1. Nearest anchor
    h_anc = nearest_anchor(h0)
    mc_ref = anchor_states[h_anc]

    # 2. Target variables (forward pass, no MCMC)
    pars = jnp.array([h0])
    tgt_vars = vs.get_state(pars).variables

    # 3. IS estimates
    H_tgt = create_operator(pars)
    r_E   = expect_is(H_tgt, mc_ref, tgt_vars)
    r_Mz2 = expect_is(Mz2,   mc_ref, tgt_vars)

    is_E.append(float(r_E.stats.mean.real))
    is_E_err.append(float(r_E.stats.error_of_mean))
    is_Mz2.append(float(r_Mz2.stats.mean.real))
    is_Mz2_err.append(float(r_Mz2.stats.error_of_mean))
    is_ess_frac.append(r_E.ess_fraction)
    is_anchor_used.append(h_anc)

print(f"IS sweep done ({len(h_sweep)} points, {len(h_anchors)} anchors).")

### 4.6 Baseline: full MCMC at every point

For comparison, we now do what one would do without IS: for each $h_0$ in the same
sweep, call `vs.get_state(h0)`, run full MCMC, and compute observables with
`MCState.expect`. 



In [ ]:
# --- warm-up ---
_mc_w = vs.get_state(jnp.array([1.0]))
_mc_w.sample()
_ = _mc_w.expect(create_operator(jnp.array([1.0])))
_ = _mc_w.expect(Mz2)

base_E, base_E_err, base_Mz2, base_Mz2_err = [], [], [], []

for h0 in h_sweep:
    pars = jnp.array([h0])
    mc = vs.get_state(pars)
    mc.sample()  # full burn-in + MCMC every time

    H_h = create_operator(pars)
    e   = mc.expect(H_h)
    mz2 = mc.expect(Mz2)

    base_E.append(float(e.mean.real))
    base_E_err.append(float(e.error_of_mean))
    base_Mz2.append(float(mz2.mean.real))
    base_Mz2_err.append(float(mz2.error_of_mean))

print(f"Baseline sweep done ({len(h_sweep)} points).")

### 4.7 Exact diagonalisation (ground truth)

Since $L=10$ is small enough for exact diagonalisation ($2^{10} = 1024$ states),
we compute the exact ground-state energy and $\langle M_z^2\rangle$ across the
sweep for comparison.


In [ ]:
from scipy.sparse import csr_matrix

Mz2_sparse = Mz2.to_sparse()
exact_E, exact_Mz2 = [], []

for h0 in h_sweep:
    H_h = create_operator(jnp.array([h0]))
    E0, psi0 = nk.exact.lanczos_ed(H_h, k=1, compute_eigenvectors=True)
    psi0 = psi0.reshape(-1)
    exact_E.append(float(E0))
    exact_Mz2.append(float(np.real(psi0.conj() @ (Mz2_sparse @ psi0))))

print("Exact diagonalisation done.")

### 4.8 Comparison plots

Three panels:

- **Energy** $\langle H\rangle$ vs $h_0$: exact (line), baseline (crosses), IS
   (circles with error bars).
- **Magnetization** $\langle M_z^2\rangle$ vs $h_0$
- **ESS / N** vs $h_0$: the IS quality diagnostic.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# --- Energy ---
ax = axes[0, 0]
ax.plot(h_sweep, exact_E, "k-", lw=1.5, label="exact")
ax.errorbar(h_sweep, base_E, yerr=base_E_err, fmt="x", ms=6, label="baseline (MCMC each)")
ax.errorbar(h_sweep, is_E, yerr=is_E_err, fmt="o", ms=5, label="IS (3 anchors)")
for h_a in h_anchors:
    ax.axvline(h_a, color="gray", ls="--", lw=0.5)
ax.set_xlabel("$h_0$"); ax.set_ylabel("$\\langle H \\rangle$")
ax.set_title("Energy"); ax.legend(fontsize=8)

# --- Mz^2 ---
ax = axes[0, 1]
ax.plot(h_sweep, exact_Mz2, "k-", lw=1.5, label="exact")
ax.errorbar(h_sweep, base_Mz2, yerr=base_Mz2_err, fmt="x", ms=6, label="baseline")
ax.errorbar(h_sweep, is_Mz2, yerr=is_Mz2_err, fmt="o", ms=5, label="IS")
for h_a in h_anchors:
    ax.axvline(h_a, color="gray", ls="--", lw=0.5)
ax.set_xlabel("$h_0$"); ax.set_ylabel("$\\langle M_z^2 \\rangle$")
ax.set_title("Magnetization squared"); ax.legend(fontsize=8)

# --- ESS ---
ax = axes[1, 0]
ax.plot(h_sweep, is_ess_frac, "s-", ms=5)
ax.axhline(0.1, color="r", ls=":", label="threshold 0.1")
for h_a in h_anchors:
    ax.axvline(h_a, color="gray", ls="--", lw=0.5)
ax.set_xlabel("$h_0$"); ax.set_ylabel("ESS / N")
ax.set_ylim(0, 1.05)
ax.set_title("Effective Sample Size"); ax.legend(fontsize=8)


plt.tight_layout(); plt.show()